In [12]:
import torch
import torch.nn as nn

In [13]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()

        self.hidden_size = hidden_size
        
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        
        # La couche de sortie 
        self.fc = nn.Linear(hidden_size, output_size)
        
        # La fonction d'activation finale
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Initialisation de l'état caché h0 à zéro
        h0 = torch.zeros(1, x.size(0), self.hidden_size)
        
        # Forward Propagation
        # out contient les outputs de chaque étape, _ contient le dernier état h4
        _, h_final = self.rnn(x, h0)
        
        # On prend le dernier état caché (Many-to-One); On le passe dans la couche de sortie
        out = self.fc(h_final.squeeze(0))
        return self.sigmoid(out)

In [14]:
# initialisation du modèle
model = SimpleRNN(input_size= 10, hidden_size= 20, output_size= 1)



In [15]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr= 0.01)

In [16]:
# Test data
phrase_test = torch.randn(1, 4, 10) 
label_reel = torch.tensor([[1.0]]) # la phrase est positive

In [17]:
for epoch in range(100):
    # Forward Propagation
    prediction = model(phrase_test)
    
    # Calcul de l'erreur
    loss = loss_fn(prediction, label_reel)
    
    # Backpropagation 
    optimizer.zero_grad() 
    loss.backward()     
    
    # Update
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Époque {epoch+1}, Erreur: {loss.item():.4f}")

Époque 10, Erreur: 0.4570
Époque 20, Erreur: 0.3622
Époque 30, Erreur: 0.2952
Époque 40, Erreur: 0.2462
Époque 50, Erreur: 0.2092
Époque 60, Erreur: 0.1806
Époque 70, Erreur: 0.1580
Époque 80, Erreur: 0.1398
Époque 90, Erreur: 0.1249
Époque 100, Erreur: 0.1125


#### Exercice

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

num_phrases = 100
longueur_phrase = 5
dimension_mot = 10


# Génération de données aléatoires (X) et de labels (y)
X_data = torch.randn(num_phrases, longueur_phrase, dimension_mot) # [100 phrases, 5 mots, 10 caractéristiques par mot]
y_data = torch.randint(0, 2, (num_phrases, 1)).float() # 100 labels : 0 OU 1


# Création du DataLoader pour gérer les Batchs
dataset = TensorDataset(X_data, y_data)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [3]:
import torch.nn as nn
import torch.optim as optim

# La classe RNN
class SentimentRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SentimentRNN, self).__init__()
        
        # initialisation du hidden_size
        self.hidden_size = hidden_size
        
        
        # Les couches RNN
        self.rnn = nn.RNN(input_size, hidden_size, batch_first= True)
        
        # Couche de sortie 
        self.fc = nn.Linear(hidden_size, output_size)
        
        # fonction d'activation
        self.sigmoid = nn.Sigmoid()
        
        
        
    # la fonction forward
    def forward(self, x):
        # initialisation de h0
        h0 = torch.zeros(1, x.size(0), self.hidden_size)
        
        # 
        out, h_final = self.rnn(x, h0)
        
        last_hidden = self.fc(h_final.squeeze(0))
        return self.sigmoid(last_hidden)

In [5]:
model = SentimentRNN(input_size= 10, hidden_size= 16, output_size= 1 )

loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr= 0.01)



In [14]:
epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = loss_fn(outputs, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Époque [{epoch+1}/{epochs}], Erreur moyenne: {total_loss/len(train_loader):.4f}")

model.eval()
test_phrase = torch.randn(1, 5, 10) 
with torch.no_grad():
    pred = model(test_phrase)
    sentiment = "Positif" if pred.item() > 0.5 else "Négatif"
    print(f"\nTest sur une phrase aléatoire : {pred.item():.4f} -> {sentiment}")

Époque [5/20], Erreur moyenne: 0.6858
Époque [10/20], Erreur moyenne: 0.6754
Époque [15/20], Erreur moyenne: 0.6735
Époque [20/20], Erreur moyenne: 0.6653

Test sur une phrase aléatoire : 0.4960 -> Négatif
